# **Environment Setup**

In [ ]:
%pip install -q -U groq

In [ ]:
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    from getpass import getpass
    GROQ_API_KEY = getpass('Paste your Groq API key (input hidden): ')

client = Groq(api_key=GROQ_API_KEY)

# Query the models actually available to this key rather than hardcoding one
available = client.models.list()
available_ids = [m.id for m in available.data]
print("Models available to your key:")
for mid in available_ids:
    print(" -", mid)

# Prefer plain instruct models over gpt-oss. gpt-oss uses OpenAI's "Harmony"
# response format, which can raise a spurious tool-call error once a system
# prompt is introduced later in this notebook.
PREFERRED_MODELS = [
    "qwen/qwen3.8-27b",
    "qwen/qwen3.6-27b",
    "qwen/qwen3-32b",
    "moonshotai/kimi-k2-instruct-0905",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
]

MODEL_ID = next((m for m in PREFERRED_MODELS if m in available_ids), None)
if MODEL_ID is None and available_ids:
    MODEL_ID = available_ids[0]

print("\nSelected MODEL_ID:", MODEL_ID)
if MODEL_ID and "gpt-oss" in MODEL_ID:
    print("Note: gpt-oss models can raise a 'tool_use_failed' error once a system")
    print("prompt is introduced later in this notebook. Pick a different MODEL_ID above if so.")

# Qwen models are hybrid "thinking" models; reasoning_effort="none" suppresses
# visible <think> tags. gpt-oss models use a different low/medium/high scale.
if MODEL_ID and MODEL_ID.startswith("qwen/"):
    REASONING_EFFORT = "none"
elif MODEL_ID and "gpt-oss" in MODEL_ID:
    REASONING_EFFORT = "low"
else:
    REASONING_EFFORT = None

sanity_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is overfitting in machine learning?"}],
    max_tokens=100,
    **sanity_kwargs,
)
print(response.choices[0].message.content)

# **Baseline Chat (No Persona)**

In [ ]:
%pip install -q -U gradio

In [ ]:
import gradio as gr
print("Gradio version:", gr.__version__)

def basic_chat(message, history):
    """Plain baseline chat: no persona, no prompting technique. Normalizes Gradio's
    history format (dict or tuple, depending on version) into {'role', 'content'}
    dicts only, since Groq rejects unrecognized extra fields."""
    messages = []
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role"), "content": turn.get("content")})
        else:
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if bot_msg:
                messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=800,
    )
    return response.choices[0].message.content


demo = gr.ChatInterface(
    fn=basic_chat,
    title="The Interrogator — Baseline Chat",
    description="Plain baseline chat, no detective persona applied.",
)

demo.launch(share=True, debug=True)

# **Persona and Few-Shot Prompting**

In [ ]:
MAX_QUESTIONS = 5  # maximum follow-up questions before a case must close

FEW_SHOT_EXAMPLES = """
--- EXAMPLE 1: A vague opener with almost no evidence yet — ask, don't guess ---
User: My cat has been acting weird lately.
Detective:
CASE OPENED

EVIDENCE:
- Your cat's behavior has changed, but no specifics are given yet.

HYPOTHESES:
- H1: A physical health issue (pain, illness).
- H2: A stress or environmental change (new pet, moved furniture, schedule change).
- H3: A normal age-related behavior shift.

MISSING:
- What specifically changed (eating, hiding, aggression, vocalizing).
- How long this has been going on.

What exactly has changed about your cat's behavior, and when did you first notice it?

--- EXAMPLE 2: A technical report with real signal — form sharp, specific hypotheses ---
User: My web app returns intermittent 500 errors, only during peak traffic hours.
Detective:
CASE OPENED

EVIDENCE:
- 500 errors occur intermittently, correlated with peak traffic.

HYPOTHESES:
- H1: Resource exhaustion under load (DB connection pool, memory, CPU).
- H2: A race condition that only surfaces with concurrent requests.
- H3: A downstream dependency (API, cache, DB) timing out under load.

MISSING:
- Whether the errors cluster around one endpoint or are spread across the app.
- What the server logs show at the moment of failure.

When these 500s happen, do your server logs point to a specific endpoint or component, or does it look random across the app?

--- EXAMPLE 3: Enough evidence in one message — close immediately, no unnecessary questions ---
User: I dropped my phone in the toilet five minutes ago and now the screen is completely black and it won't turn on.
Detective:
CASE CLOSED

PRIMARY CONCLUSION: Water damage from the toilet drop has short-circuited or shut down your phone.

EVIDENCE: Direct liquid exposure five minutes ago, immediately followed by total screen failure — a textbook liquid-damage signature.

HYPOTHESES RULED OUT: A pre-existing battery or software fault is far less likely given the precise timing right after submersion.

RECOMMENDATION: Power it off if it isn't already, do not charge it, remove the case and SIM if possible, and leave it in a desiccant/silica-gel drying pouch for at least 24-48 hours before attempting to power on.
"""

SYSTEM_PROMPT = """You are THE INTERROGATOR, a sharp, methodical detective persona.

You do not answer questions directly. Instead, you treat every user message as a CASE.
Your job is to investigate the case, not to immediately solve it.

Behavior rules:
1. Every single response, without exception, must use these exact section
   headers in this order: for a new or ongoing case use CASE OPENED (first
   turn only) or CASE UPDATE (later turns), then EVIDENCE:, HYPOTHESES:,
   MISSING:, and end with your one question. Use these headers even for
   short, casual, or low-detail messages — do not drop the structure just
   because there isn't much evidence yet; in that case, say so explicitly
   under EVIDENCE: and MISSING:.
2. Identify the EVIDENCE already present in what the user told you.
3. Form one or more plausible HYPOTHESES based on that evidence.
4. Identify what key information is still MISSING.
5. Ask exactly ONE focused, high-value follow-up question to fill the most
   important gap. Never ask multiple questions in the same turn.
6. Continue this investigation across turns: update evidence and hypotheses
   as the user answers, and ask the next most useful question.
7. You will be told in a status note how many questions you have already
   asked. You may ask at most {max_q} follow-up questions total. Once that
   limit is reached (the status note will say so explicitly), you MUST NOT
   ask another question — you must close the case on that turn instead.
8. To close a case: say "CASE CLOSED," state your primary conclusion,
   briefly note the evidence that supports it, and mention any hypotheses
   you ruled out and why. Keep the closing summary tight — a few sentences
   per section, not an essay.
9. Stay in character. Use light detective language (case, evidence,
   hypothesis, lead, clue) naturally, but don't overdo it to the point of
   being hard to read.
10. If a case has already been closed and the user sends a new, clearly
    different problem to investigate, open a brand-new case from scratch
    (do not blend it with the old one). If the user is just acknowledging,
    thanking you, or confirming they'll follow your advice — for example
    "thanks", "okay", "got it", "okay i will do that", "sure" — that is
    NOT a new case, even though it sounds like agreement or future action.
    Reply briefly and naturally in character instead of opening a case.
    If you are genuinely unsure whether a message is a new case or just an
    acknowledgment, default to treating it as an acknowledgment.
11. Always address the person directly as "you." Never refer to them in
    the third person as "the user" — that breaks the conversational feel.
    Example: write "What were you doing right before it started?" not
    "What was the user doing before the headache started?"
12. The examples below show the expected style, structure, and judgment —
    including when to ask a question versus when there's already enough
    evidence to close immediately. Match this style and level of specificity,
    but never reuse or reference the example cases themselves; each new
    case from the user is independent.

EXAMPLES:
{few_shot}
""".format(max_q=MAX_QUESTIONS, few_shot=FEW_SHOT_EXAMPLES)


def extract_text(content):
    """Normalizes Gradio message content to plain text. Content can be a string
    or a list of parts (e.g. [{'type': 'text', 'text': '...'}]) depending on
    the Gradio version in use."""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
        return "".join(parts)
    return "" if content is None else str(content)


def normalize_history(history):
    """Flattens Gradio history (dict or tuple format) into a list of (role, text) tuples."""
    normalized = []
    for turn in history:
        if isinstance(turn, dict):
            normalized.append((turn.get("role"), extract_text(turn.get("content"))))
        else:
            user_msg, bot_msg = turn
            if user_msg:
                normalized.append(("user", extract_text(user_msg)))
            if bot_msg:
                normalized.append(("assistant", extract_text(bot_msg)))
    return normalized


def count_questions_asked(history):
    """Counts assistant turns since the most recent case closure, so a new case
    does not inherit an already-exhausted question budget from a prior case."""
    normalized = normalize_history(history)

    last_close_idx = -1
    for i, (role, text) in enumerate(normalized):
        if role == "assistant" and "CASE CLOSED" in text.upper():
            last_close_idx = i

    count = 0
    for i, (role, text) in enumerate(normalized):
        if i > last_close_idx and role == "assistant":
            count += 1
    return count


def case_already_closed(history):
    """Returns whether the case is currently closed, based on the most recent
    assistant turn that carried a CASE OPENED/UPDATE/CLOSED header. Header-less
    acknowledgment replies (e.g. "glad I could help") do not change this state,
    which avoids mistaking small talk after a close for the case still being open."""
    state_closed = False
    for role, text in normalize_history(history):
        if role != "assistant":
            continue
        upper = text.upper().lstrip()
        if upper.startswith("CASE CLOSED"):
            state_closed = True
        elif upper.startswith("CASE OPENED") or upper.startswith("CASE UPDATE"):
            state_closed = False
    return state_closed


def history_since_last_close(history):
    """Returns the turns after the most recent CASE CLOSED, excluding any earlier,
    already-closed case from the same session."""
    normalized = normalize_history(history)
    last_close_idx = -1
    for i, (role, text) in enumerate(normalized):
        if role == "assistant" and text.upper().lstrip().startswith("CASE CLOSED"):
            last_close_idx = i
    return normalized[last_close_idx + 1:]


def interrogator_chat(message, history):
    try:
        return _interrogator_chat_inner(message, history)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        return "The precinct's lines are jammed right now. Give it a minute and try again."


def _interrogator_chat_inner(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn.get("role"), "content": extract_text(turn.get("content"))})
        else:
            user_msg, bot_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": extract_text(user_msg)})
            if bot_msg:
                messages.append({"role": "assistant", "content": extract_text(bot_msg)})

    # The status note is folded into the user turn as plain prose rather than sent
    # as a second system message, since some models misread an extra system-role
    # message as a tool invocation.
    if case_already_closed(history):
        status_note = (
            "(Detective's internal note: the previous case is already closed. "
            "If this new message presents a genuinely new, different problem to "
            "investigate, open a brand-new case and start from scratch. But phrases "
            "like 'thanks', 'okay', 'got it', or 'okay I will do that' are just "
            "acknowledgments or confirmations, NOT new cases, even though they "
            "sound like agreement or future action. For those, reply briefly and "
            "naturally in character, WITHOUT opening a case or repeating the old "
            "closing report. If unsure, default to treating it as an acknowledgment.)\n\n"
        )
    else:
        questions_asked = count_questions_asked(history)
        remaining = max(MAX_QUESTIONS - questions_asked, 0)
        if remaining == 0:
            status_note = (
                f"(Detective's internal note: you have already asked {questions_asked} questions, "
                f"the maximum allowed. You must close the case now instead of asking another question.)\n\n"
            )
        else:
            status_note = (
                f"(Detective's internal note: you have asked {questions_asked} question so far, "
                f"you may ask up to {remaining} more if needed, or close the case now if you're confident.)\n\n"
            )

    messages.append({"role": "user", "content": status_note + extract_text(message)})

    extra_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=900,
        temperature=0.4,
        **extra_kwargs,
    )
    return response.choices[0].message.content


demo = gr.ChatInterface(
    fn=interrogator_chat,
    title="The Interrogator",
    description="Present your case. The detective will investigate before concluding.",
)

demo.launch(share=True, debug=True)

# **Prompt Chaining**

In [ ]:
def clean_stage_output(text):
    """Strips a leading echoed header (e.g. the model repeating 'EVIDENCE:' at the
    start of its own answer) so the final message doesn't end up with a doubled header."""
    text = text.strip()
    known_headers = ["EVIDENCE:", "HYPOTHESES:", "MISSING:", "QUESTION:", "CONCLUSION:", "DECISION:"]
    for header in known_headers:
        if text.upper().startswith(header):
            text = text[len(header):].lstrip(" \n")
            break
    return text


def call_stage(user_content, max_tokens=200):
    """One link in the prompt chain: a single-purpose call with no conversation
    history, just the specific context that stage needs."""
    extra_kwargs = {"reasoning_effort": REASONING_EFFORT} if REASONING_EFFORT else {}
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "user", "content": user_content}],
        max_tokens=max_tokens,
        temperature=0.4,
        **extra_kwargs,
    )
    return response.choices[0].message.content.strip()


def build_convo_text(context_turns):
    """Joins a list of (role, text) tuples into plain 'Role: text' lines. The
    caller decides what context_turns contains — see chained_interrogator_chat."""
    return "\n".join(f"{role.capitalize()}: {text}" for role, text in context_turns)


# --- Stage 1: Evidence Extraction ---
def extract_evidence(context_turns, message):
    """context_turns: the relevant prior turns for the CURRENT case only. Pass an
    empty list for a brand-new case, or history_since_last_close(history) for a
    continuing case, so an unrelated old case can't leak in as evidence."""
    convo_text = build_convo_text(context_turns)
    prompt = (
        "You are the EVIDENCE-EXTRACTION stage of a detective's case-analysis pipeline.\n"
        "Read the ENTIRE conversation so far, plus the newest message. Produce ONE cumulative "
        "list of ALL explicit, concrete facts stated about the situation across the WHOLE case "
        "so far — combine facts from earlier turns with anything new in this message. Do NOT "
        "drop or forget facts from earlier turns just because the newest message doesn't repeat "
        "them. No hypotheses, no psychological speculation about the person's mental or emotional "
        "state, no missing-information notes. Address the person as 'you', never 'the speaker' "
        "or 'the user'. Keep each bullet ONE short sentence, maximum 6 bullets. Do NOT include "
        "the word 'EVIDENCE' as a header in your answer — start directly with the first bullet.\n\n"
        f"CONVERSATION SO FAR:\n{convo_text if convo_text else '(none yet)'}\n\n"
        f"NEWEST MESSAGE: {message}\n\nEVIDENCE:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=280))


# --- Stage 2: Hypothesis Generation ---
def generate_hypotheses(evidence_text):
    prompt = (
        "You are the HYPOTHESIS-GENERATION stage of a detective's case-analysis pipeline.\n"
        "Given ONLY this evidence, list 2-4 plausible, SPECIFIC and CONCRETE hypotheses about "
        "the practical SITUATION or PROBLEM described. Do NOT psychoanalyze the person's "
        "emotional or mental state as the hypothesis itself — focus on the concrete problem, "
        "not a personality profile. Address the person as 'you', never 'the speaker'. Keep each "
        "bullet ONE short sentence, no sub-explanations. Bulleted list only, no other commentary.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=280))


# --- Stage 3: Evidence Gap Detection ---
def find_missing(evidence_text, hypotheses_text):
    prompt = (
        "You are the EVIDENCE-GAP stage of a detective's case-analysis pipeline.\n"
        "Given this evidence and these hypotheses, identify the 1-3 most useful pieces of "
        "missing information that would help distinguish between the hypotheses. Address the "
        "person as 'you'. Keep each bullet ONE short sentence, no sub-explanations. Bulleted "
        "list only.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nMISSING:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=220))


# --- Stage 4: Decision (close vs. continue) ---
def decide_close(evidence_text, hypotheses_text, missing_text, questions_asked, max_questions):
    prompt = (
        "You are the DECISION stage of a detective's case-analysis pipeline.\n"
        f"So far {questions_asked} follow-up question(s) have been asked (max allowed: {max_questions}).\n"
        "Given the evidence, hypotheses, and missing information below, decide: is there already "
        "enough evidence to confidently close the case, or is one more focused question needed?\n"
        "Answer with EXACTLY one word: CLOSE or CONTINUE.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nMISSING:\n{missing_text}\n\nDECISION:"
    )
    decision = call_stage(prompt, max_tokens=5).strip().upper()
    return "CLOSE" if "CLOSE" in decision else "CONTINUE"


# --- Stage 5a: Next Question ---
def generate_question(evidence_text, hypotheses_text, missing_text):
    prompt = (
        "You are the NEXT-QUESTION stage of a detective's case-analysis pipeline.\n"
        "Given the evidence, hypotheses, and missing information below, write exactly ONE "
        "focused, high-value follow-up question, addressed directly to the person as 'you' "
        "(never 'the user'). Return ONLY the question, nothing else.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nMISSING:\n{missing_text}\n\nQUESTION:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=100))


# --- Stage 5b: Case Conclusion ---
def generate_conclusion(evidence_text, hypotheses_text):
    prompt = (
        "You are the CASE-CONCLUSION stage of a detective's case-analysis pipeline.\n"
        "Given the evidence and hypotheses below, synthesize a final case conclusion with these "
        "EXACT section headers: PRIMARY CONCLUSION, EVIDENCE, HYPOTHESES RULED OUT, RECOMMENDATION. "
        "Keep it tight — a few sentences per section, addressed directly to 'you'.\n\n"
        f"EVIDENCE:\n{evidence_text}\n\nHYPOTHESES:\n{hypotheses_text}\n\nCONCLUSION:"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=400))


# --- Post-closure small talk: kept as a single lightweight call, not the full chain ---
def classify_after_close(message):
    # Short, common acknowledgments are classified directly, without an LLM call,
    # since a model classifier is least reliable on very short, ambiguous input.
    trivial_acks = {
        "ok", "okay", "k", "kk", "alright", "sure", "thanks", "thank you", "cool",
        "got it", "sounds good", "noted", "fine", "yep", "yup", "great", "nice",
    }
    if message.strip().lower().rstrip("!.") in trivial_acks:
        return "ACK"

    prompt = (
        "The previous case was already closed. Classify this new user message as EXACTLY one word:\n"
        "ACK — just an acknowledgment, thanks, or confirmation like 'okay', 'thanks', 'got it', "
        "'okay i will do that', 'sure' (even if it sounds like future action, it is NOT a new case).\n"
        "NEW_CASE — a genuinely new, different problem to investigate.\n"
        "If unsure, default to ACK.\n\n"
        f'Message: "{message}"\n\nAnswer with exactly one word (ACK or NEW_CASE):'
    )
    answer = call_stage(prompt, max_tokens=5).strip().upper()
    return "NEW_CASE" if "NEW_CASE" in answer else "ACK"


def brief_ack_reply(message):
    prompt = (
        "You are THE INTERROGATOR, a sharp detective persona, speaking casually now that the case "
        "is closed. The user just sent a brief acknowledgment or thanks. Reply briefly (1-2 sentences), "
        "in character, without opening a new case or repeating any case report.\n\n"
        f'User said: "{message}"\n\nYour brief in-character reply:'
    )
    return call_stage(prompt, max_tokens=80)


def chained_interrogator_chat(message, history):
    try:
        return _chained_interrogator_chat_inner(message, history)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        return "The precinct's lines are jammed right now. Give it a minute and try again."


def _chained_interrogator_chat_inner(message, history):
    message = extract_text(message)

    if case_already_closed(history):
        if classify_after_close(message) == "ACK":
            return brief_ack_reply(message)
        # Otherwise: a genuinely new case. Fall through and run the chain as CASE OPENED.

    is_new_case = (len(history) == 0) or case_already_closed(history)

    # A brand-new case starts with no prior context at all, not even "since the last
    # close" — that window can still contain the small-talk exchange that followed
    # the old case's closure, which is not evidence for the new case.
    evidence_context = [] if is_new_case else history_since_last_close(history)

    print("--- Stage 1: Evidence Extraction ---")
    evidence = extract_evidence(evidence_context, message)
    print(evidence)

    print("\n--- Stage 2: Hypothesis Generation ---")
    hypotheses = generate_hypotheses(evidence)
    print(hypotheses)

    print("\n--- Stage 3: Evidence Gap ---")
    missing = find_missing(evidence, hypotheses)
    print(missing)

    questions_asked = count_questions_asked(history)
    remaining = max(MAX_QUESTIONS - questions_asked, 0)
    decision = "CLOSE" if remaining == 0 else decide_close(evidence, hypotheses, missing, questions_asked, MAX_QUESTIONS)
    print(f"\n--- Stage 4: Decision --- {decision}")

    if decision == "CLOSE":
        print("\n--- Stage 5b: Case Conclusion ---")
        conclusion = generate_conclusion(evidence, hypotheses)
        print(conclusion)
        return f"CASE CLOSED\n\n{conclusion}"
    else:
        print("\n--- Stage 5a: Next Question ---")
        question = generate_question(evidence, hypotheses, missing)
        print(question)
        header = "CASE OPENED" if is_new_case else "CASE UPDATE"
        return (
            f"{header}\n\n"
            f"EVIDENCE:\n{evidence}\n\n"
            f"HYPOTHESES:\n{hypotheses}\n\n"
            f"MISSING:\n{missing}\n\n"
            f"{question}"
        )


demo = gr.ChatInterface(
    fn=chained_interrogator_chat,
    title="The Interrogator — Prompt-Chained",
    description="Present your case. Each response is built from 4-6 separate staged reasoning calls — check the cell's log output below to see each stage.",
)

demo.launch(share=True, debug=True)

# **Structured Case State**

In [ ]:
def new_case_state():
    """A fresh, empty case. Evidence accumulates into this via list.append() as the
    case progresses, instead of re-deriving the whole list from scratch each turn."""
    return {
        "status": "OPEN",
        "evidence": [],
        "hypotheses": "",
        "missing": "",
        "questions_asked": 0,
    }


def parse_bullets(text):
    """Splits a bulleted LLM response into a clean list of strings, dropping
    bullet markers and blank lines."""
    lines = []
    for line in text.split("\n"):
        line = line.strip().lstrip("*-•").strip()
        if line:
            lines.append(line)
    return lines


def extract_new_evidence(message):
    """Extracts only the new facts from this single message, not the case's full
    cumulative evidence. The caller appends the result to case_state['evidence'],
    so the model never has to repeat facts from earlier turns."""
    prompt = (
        "Extract ONLY the new explicit, concrete facts stated in this single message — "
        "no hypotheses, no psychological speculation, no missing-info notes. Address the "
        "person as 'you'. One short sentence per bullet, maximum 3 bullets. If the message "
        "has no new concrete facts (e.g. it's vague or just an exclamation), return nothing.\n\n"
        f"MESSAGE: {message}\n\nNEW FACTS (bulleted, or leave blank if none):"
    )
    return clean_stage_output(call_stage(prompt, max_tokens=120))


def stateful_investigate(message, case_state):
    """Public entry point: catches any stage failure (e.g. an API rate limit
    mid-investigation) and degrades gracefully instead of crashing the app.
    case_state is left untouched on failure, so retrying continues where it left off."""
    try:
        return _stateful_investigate_inner(message, case_state)
    except Exception as e:
        print(f"[stage error] {type(e).__name__}: {e}")
        friendly = (
            "The precinct's lines are jammed right now. Give it a minute and send "
            "that again; nothing about your case has been lost."
        )
        return friendly, case_state


def _stateful_investigate_inner(message, case_state):
    """Reads and writes an explicit case_state dict instead of re-inferring case
    status, evidence, and question count from raw chat history each turn."""
    message = extract_text(message)

    if case_state["status"] == "CLOSED":
        if classify_after_close(message) == "ACK":
            return brief_ack_reply(message), case_state
        case_state = new_case_state()

    is_new_case = (len(case_state["evidence"]) == 0)

    print("--- Stage 1: New-Evidence Extraction ---")
    new_facts_text = extract_new_evidence(message)
    new_facts = parse_bullets(new_facts_text)
    case_state["evidence"].extend(new_facts)
    print(new_facts if new_facts else "(no new concrete facts)")

    evidence_text = "\n".join(f"- {e}" for e in case_state["evidence"]) or "(none yet)"

    print("\n--- Stage 2: Hypothesis Generation ---")
    case_state["hypotheses"] = generate_hypotheses(evidence_text)
    print(case_state["hypotheses"])

    print("\n--- Stage 3: Evidence Gap ---")
    case_state["missing"] = find_missing(evidence_text, case_state["hypotheses"])
    print(case_state["missing"])

    remaining = max(MAX_QUESTIONS - case_state["questions_asked"], 0)
    decision = "CLOSE" if remaining == 0 else decide_close(
        evidence_text, case_state["hypotheses"], case_state["missing"],
        case_state["questions_asked"], MAX_QUESTIONS,
    )
    print(f"\n--- Stage 4: Decision --- {decision}")

    if decision == "CLOSE":
        print("\n--- Stage 5b: Case Conclusion ---")
        conclusion = generate_conclusion(evidence_text, case_state["hypotheses"])
        print(conclusion)
        case_state["status"] = "CLOSED"
        reply = f"CASE CLOSED\n\n{conclusion}"
    else:
        print("\n--- Stage 5a: Next Question ---")
        question = generate_question(evidence_text, case_state["hypotheses"], case_state["missing"])
        print(question)
        case_state["questions_asked"] += 1
        header = "CASE OPENED" if is_new_case else "CASE UPDATE"
        reply = (
            f"{header}\n\n"
            f"EVIDENCE:\n{evidence_text}\n\n"
            f"HYPOTHESES:\n{case_state['hypotheses']}\n\n"
            f"MISSING:\n{case_state['missing']}\n\n"
            f"{question}"
        )

    return reply, case_state


# gr.Blocks + gr.State carries case_state across turns, which gr.ChatInterface
# (used in the previous two sections) does not support.
with gr.Blocks(title="The Interrogator — Structured Case State") as demo:
    gr.Markdown("### The Interrogator — Structured Case State\nPresent your case. Case status, evidence, and question count are tracked explicitly, not re-inferred from chat text each turn.")
    chatbot = gr.Chatbot(height=450)
    case_state_box = gr.State(new_case_state())
    msg = gr.Textbox(label="Your message", placeholder="Present your case...")
    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        new_case_btn = gr.Button("New Case")

    def handle_send(message, chat_history, case_state):
        if not message or not message.strip():
            return chat_history, case_state, ""
        reply, case_state = stateful_investigate(message, case_state)
        chat_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": reply},
        ]
        return chat_history, case_state, ""

    def handle_new_case(chat_history, case_state):
        """Clears the visible chat as well as the internal state, so there is a
        clear line between the old case and the new one."""
        fresh_chat = [{"role": "assistant", "content": "New case started. What's going on?"}]
        return fresh_chat, new_case_state()

    msg.submit(handle_send, [msg, chatbot, case_state_box], [chatbot, case_state_box, msg])
    send_btn.click(handle_send, [msg, chatbot, case_state_box], [chatbot, case_state_box, msg])
    new_case_btn.click(handle_new_case, [chatbot, case_state_box], [chatbot, case_state_box])

demo.launch(share=True, debug=True)